# Manutenção preditiva industrial

Paradas inesperadas de equipamentos afetam a continuidade de uma linha de produção. Neste projeto, usamos registros históricos de operação e medições de sensores para investigar a previsão de falhas mecânicas.

O objetivo é comparar dois classificadores, KNN e Árvore de Decisão, para prever `falha_maquina`: `0` representa funcionamento normal e `1` representa falha. O trabalho percorre o entendimento dos dados, sua preparação, a construção de uma variável, o treinamento e a avaliação dos modelos.

A base utilizada está em `data/manutencao_preditiva.csv`. As anotações do Departamento de Engenharia orientam a interpretação das colunas e a identificação das informações que só ficam disponíveis após uma falha.

## Contrato de dados

Antes de analisar ou treinar modelos, precisamos definir o significado e o papel de cada coluna. A tabela abaixo registra, para cada variável original, se seu valor estaria disponível no momento da previsão e se ela será candidata a preditora (`X`), alvo (`y`) ou excluída das preditoras.

“Excluída” significa fora de `X`; a coluna pode permanecer na base original para consulta e auditoria.



| Variável | Significado | Disponível antes da falha? | Papel proposto (X/y/excluído) | Justificativa |
|-|-|-|-|-|
| `udi` | Identificador único numérico para cada registro na base de dados (varia de 1 a 10.000). | Sim | Excluir | Identifica a linha, não uma condição operacional; sua sequência pode induzir associações artificiais. |
| `id_produto` | Código de identificação exclusivo do produto, composto por uma letra que indica a qualidade e um número de série. | Sim | Excluir | Sem evidência de sinal generalizável, devemos excluir pois pode induzir o modelo a memorização de padrões. |
| `tipo` | Categoria do equipamento baseada na sua especificação técnica, sendo dividido em três classes: L (Low - Baixo), M (Medium - Médio) ou H (High - Alto). | Sim | X | Categoria conhecida antes da falha; representá-la por colunas indicadoras evita atribuir distâncias numéricas arbitrárias às categorias. |
| `temperatura_ar_k` | Temperatura do ambiente onde a máquina está instalada, medida na escala Kelvin. | Sim | X | A temperatura do ar compõe as condições de dissipação de calor. Sua relação com falhas deve ser investigada junto à temperatura do processo e à rotação, sem presumir causalidade isolada.  |
| `temperatura_processo_k` | Temperatura gerada durante a operação do processo de fabricação, medida na escala Kelvin. | Sim | X | A diferença em relação à temperatura do ar pode influenciar a dissipação térmica durante a operação; trata-se de uma medida disponível para avaliar o estado do processo. |
| `velocidade_rotacao_rpm` | Velocidade de giro do motor do equipamento, medida em Rotações Por Minuto (RPM). | Sim | X | A rotação caracteriza o regime de operação e, junto ao torque, relaciona-se à potência mecânica requerida pelo processo. |
| `torque_nm` | Momento de força gerada pelo motor da máquina, medida em Newton-metro (N.m). | Sim | X | O torque, em conjunto com a rotação, relaciona-se à potência mecânica; em conjunto com o tempo de uso da ferramenta, ajuda a descrever o esforço aplicado. |
| `desgaste_ferramenta_min` | Tempo acumulado de utilização da ferramenta de corte atual, medido em minutos. | Sim | X | Representa o tempo acumulado de uso da ferramenta, relevante para investigar desgaste e esforço mecânico. Minutos de uso não equivalem a uma medição direta do desgaste físico. |
| `falha_maquina` | Variável Alvo (Target) do projeto. Indica o estado de operação do equipamento, onde 1 representa que a máquina sofreu uma falha mecânica e 0 representa que a máquina operou normalmente. | Não | y | É o resultado que queremos prever. Incluí-lo em X revelaria diretamente a resposta ao modelo. |
| `falha_twf` | Falha por desgaste da ferramenta (Tool Wear Failure - 1 para sim, 0 para não). | Não | Excluir | Modo de falha registrado após o evento; incluí-lo em X revelaria informação sobre o alvo. |
| `falha_hdf` | Falha por dissipação de calor/superaquecimento (Heat Dissipation Failure - 1 para sim, 0 para não). | Não | Excluir | Modo de falha registrado após o evento; incluí-lo em X revelaria informação sobre o alvo. |
| `falha_pwf` | Falha por falta ou excesso de potência elétrica (Power Failure - 1 para sim, 0 para não). | Não | Excluir | Modo de falha registrado após o evento; incluí-lo em X revelaria informação sobre o alvo. |
| `falha_osf` | Falha por esforço ou tensão mecânica excessiva (Overstrain Failure - 1 para sim, 0 para não). | Não | Excluir | Modo de falha registrado após o evento; incluí-lo em X revelaria informação sobre o alvo. |
| `falha_rnf` | Falha aleatória não detectada pelos sensores padrão (Random Failure - 1 para sim, 0 para não). | Não | Excluir | Modo de falha registrado após o evento; incluí-lo em X revelaria informação sobre o alvo. |

### Critério temporal e proposta de preparação

Para este projeto, assumimos que a categoria do equipamento e as medidas operacionais estão disponíveis no momento da previsão, antes de conhecermos o resultado da falha. A base não informa quanto tempo separa as medições do evento. `falha_maquina` será a variável alvo (`y`). As cinco colunas que identificam motivos da falha ficam fora das preditoras (`X`), pois contêm informações conhecidas após o evento e causariam vazamento de dados.

As variáveis candidatas a `X` são `tipo`, as duas temperaturas, velocidade de rotação, torque e tempo de uso da ferramenta. Representaremos as três categorias de `tipo` por três colunas indicadoras. `udi` e `id_produto` serão preservados para rastrear os registros, mas não entrarão em `X`: são identificadores, sem sinal operacional generalizável demonstrado para este modelo. Nenhum deles identifica uma máquina física específica.

Planejamos dividir os dados em 80% para treino e 20% para teste, com estratificação por `y` para preservar aproximadamente a proporção original das classes em ambos os conjuntos. Se houver valores ausentes, escolheremos média ou mediana com base na distribuição observada; o valor de imputação será calculado no treino e aplicado ao teste. O balanceamento ocorrerá somente no treino. Para o KNN, o escalonamento das variáveis contínuas será ajustado no treino e aplicado ao teste, pois o algoritmo compara distâncias. A Árvore de Decisão usará essas variáveis sem escalonamento, pois suas divisões dependem de limites dos atributos, não de suas magnitudes relativas.